# Bhav Copy Parquet Data Analysis

This notebook analyzes NSE Bhav Copy data stored in Parquet format. It uses modular helper functions from the `helpers` folder for data loading, preprocessing, analysis, and reporting.

**Notebook Structure:**
1. Import Required Libraries
2. Load Bhav Copy Parquet Data
3. Explore Data Structure and Schema
4. Data Quality Checks
5. Statistical Analysis
6. Time Series Analysis
7. Volume and Price Trends
8. Generate Summary Reports

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import custom helper modules
import sys
sys.path.append(str(Path.cwd() / 'helpers'))

from data_loader import load_parquet_files, get_file_info
from data_explorer import explore_data_structure, display_column_statistics
from data_quality import check_data_quality, detect_outliers
from analysis import calculate_statistics, calculate_correlations, analyze_distributions
from time_series import analyze_trends, calculate_seasonality
from volume_analysis import analyze_volume_patterns, analyze_price_movements
from reporting import generate_summary_report, export_results

# Configure visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries and helper modules imported successfully!")

## 2. Load Bhav Copy Parquet Data

Load parquet data files from the specified directory. The helper function will automatically detect and load all parquet files.

In [ ]:
# Define the path to parquet data files
# Adjust the path based on your actual data location
parquet_data_path = Path("data/bhav_copy")

# Ensure the path exists
if not parquet_data_path.exists():
    print(f"⚠ Path does not exist: {parquet_data_path}")
    print("Please ensure parquet files are in the 'data/bhav_copy' directory")
else:
    # Load all parquet files
    df = load_parquet_files(parquet_data_path)
    print(f"✓ Data loaded successfully!")
    print(f"✓ Total records: {len(df):,}")
    print(f"✓ Date range: {df.index.min()} to {df.index.max()}" if hasattr(df.index, 'min') else "")

In [ ]:
# Display file information
file_info = get_file_info(parquet_data_path)
print("\n📁 Parquet Files Information:")
print("=" * 80)
for info in file_info:
    print(f"File: {info['filename']}")
    print(f"  Size: {info['size']:.2f} MB")
    print(f"  Records: {info['records']:,}")
    print()

print(f"Total size: {sum(f['size'] for f in file_info):.2f} MB")
print(f"Total records: {sum(f['records'] for f in file_info):,}")

## 3. Explore Data Structure and Schema

Examine the DataFrame schema, column names, data types, and basic statistics.

In [ ]:
# Display data structure
print("📊 Data Structure Overview:")
print("=" * 80)
explore_data_structure(df)

In [ ]:
# Display first few rows
print("\n📋 First 5 Rows:")
print("=" * 80)
print(df.head())

In [ ]:
# Display column statistics
print("\n📈 Column Statistics:")
print("=" * 80)
display_column_statistics(df)

In [ ]:
# Display data types summary
print("\n🔍 Data Types Summary:")
print("=" * 80)
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 4. Data Quality Checks

Perform comprehensive data quality validation including null values, duplicates, and outlier detection.

In [ ]:
# Perform data quality checks
print("🔎 Data Quality Assessment:")
print("=" * 80)
quality_report = check_data_quality(df)
print(quality_report)

In [ ]:
# Visualize missing data
print("\n📊 Missing Data Visualization:")
print("=" * 80)

fig, ax = plt.subplots(figsize=(12, 6))
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if len(missing_data) > 0:
    missing_data.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Number of Missing Values')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.show()
else:
    print("✓ No missing values detected!")

In [ ]:
# Detect outliers in numerical columns
print("\n🎯 Outlier Detection:")
print("=" * 80)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numerical_cols) > 0:
    outlier_report = detect_outliers(df, numerical_cols)
    print(outlier_report)
else:
    print("No numerical columns found for outlier detection.")

## 5. Statistical Analysis

Calculate descriptive statistics, correlations, and distributions for price and volume metrics.

In [ ]:
# Calculate descriptive statistics
print("📊 Descriptive Statistics:")
print("=" * 80)
stats_report = calculate_statistics(df)
print(stats_report)

In [ ]:
# Calculate correlations
print("\n🔗 Correlation Analysis:")
print("=" * 80)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numerical_cols) > 1:
    correlation_matrix = df[numerical_cols].corr()
    
    # Display correlation matrix
    print("\nCorrelation Matrix:")
    print(correlation_matrix)
    
    # Visualize correlation heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                fmt='.2f', square=True, ax=ax)
    ax.set_title('Correlation Matrix - Bhav Copy Data')
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient numerical columns for correlation analysis.")

In [ ]:
# Analyze distributions
print("\n📉 Distribution Analysis:")
print("=" * 80)

distribution_report = analyze_distributions(df)
print(distribution_report)

In [ ]:
# Visualize distributions for key numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()[:6]

if len(numerical_cols) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, col in enumerate(numerical_cols):
        axes[idx].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
    
    # Hide unused subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

## 6. Time Series Analysis

Analyze temporal patterns, trends, and seasonality in the data.

In [ ]:
# Analyze trends
print("📈 Time Series Trend Analysis:")
print("=" * 80)

# Identify date column (assuming index or a date column exists)
trend_report = analyze_trends(df)
print(trend_report)

In [ ]:
# Calculate seasonality
print("\n🔄 Seasonality Analysis:")
print("=" * 80)

if hasattr(df.index, 'dayofweek') or any(col in df.columns for col in ['date', 'Date', 'DATE']):
    seasonality_report = calculate_seasonality(df)
    print(seasonality_report)
else:
    print("Date index not found. Ensure data has a datetime index for seasonality analysis.")

In [ ]:
# Visualize time series trends (if applicable)
# This example assumes a date index and numerical columns

try:
    if hasattr(df.index, 'date'):
        # Get the first numerical column
        numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numerical_cols) > 0:
            fig, ax = plt.subplots(figsize=(15, 6))
            col = numerical_cols[0]
            
            df[col].plot(ax=ax, linewidth=1.5, color='steelblue')
            ax.set_title(f'Time Series: {col}')
            ax.set_xlabel('Date')
            ax.set_ylabel(col)
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
except Exception as e:
    print(f"⚠ Could not plot time series: {e}")
    print("Ensure the DataFrame has a datetime index.")

## 7. Volume and Price Trends

Analyze trading volume patterns and price movements across different time periods and securities.

In [ ]:
# Analyze volume patterns
print("📊 Volume Patterns Analysis:")
print("=" * 80)

volume_report = analyze_volume_patterns(df)
print(volume_report)

In [ ]:
# Analyze price movements
print("\n💹 Price Movements Analysis:")
print("=" * 80)

price_report = analyze_price_movements(df)
print(price_report)

In [ ]:
# Visualize volume trends
try:
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Find volume-related columns
    volume_cols = [col for col in numerical_cols if 'vol' in col.lower() or 'volume' in col.lower()]
    price_cols = [col for col in numerical_cols if 'close' in col.lower() or 'price' in col.lower()]
    
    if len(volume_cols) > 0:
        fig, ax = plt.subplots(figsize=(15, 6))
        col = volume_cols[0]
        df[col].plot(ax=ax, kind='line', linewidth=1, color='darkgreen', alpha=0.7)
        ax.set_title(f'Trading Volume Trend: {col}')
        ax.set_xlabel('Date')
        ax.set_ylabel('Volume')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"⚠ Could not visualize volume trends: {e}")

In [ ]:
# Top securities by volume
try:
    if 'symbol' in df.columns or 'Symbol' in df.columns:
        symbol_col = 'symbol' if 'symbol' in df.columns else 'Symbol'
        volume_cols = [col for col in df.columns if 'vol' in col.lower()]
        
        if len(volume_cols) > 0:
            print("\n🏆 Top 10 Securities by Total Volume:")
            print("=" * 80)
            top_volume = df.groupby(symbol_col)[volume_cols[0]].sum().nlargest(10)
            print(top_volume)
except Exception as e:
    print(f"⚠ Could not analyze top securities: {e}")

## 8. Generate Summary Reports

Create comprehensive summary reports and export results.

In [ ]:
# Generate comprehensive summary report
print("📋 Generating Summary Report:")
print("=" * 80)

summary_report = generate_summary_report(df)
print(summary_report)

In [ ]:
# Export results to CSV and Excel
print("\n💾 Exporting Analysis Results:")
print("=" * 80)

# Create output directory
output_dir = Path("output/bhav_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

# Export summary statistics
results = export_results(df, output_dir)

print(f"\n✓ Results exported to: {output_dir}")
for result in results:
    print(f"  ✓ {result}")

In [ ]:
# Display export summary
print("\n📊 Export Summary:")
print("=" * 80)

if output_dir.exists():
    files = list(output_dir.glob('*'))
    if len(files) > 0:
        print(f"Total files created: {len(files)}\n")
        for file in files:
            size_mb = file.stat().st_size / (1024**2)
            print(f"  📄 {file.name} ({size_mb:.2f} MB)")
    else:
        print("No files were created during export.")
else:
    print("Output directory not found.")

## Analysis Complete ✓

This notebook has performed a comprehensive analysis of the Bhav Copy parquet data including:

- **Data Loading**: Loaded and inspected parquet files
- **Data Exploration**: Examined structure, schema, and statistics
- **Quality Checks**: Validated data integrity and identified outliers
- **Statistical Analysis**: Calculated descriptive statistics and correlations
- **Time Series Analysis**: Analyzed temporal patterns and seasonality
- **Volume & Price Analysis**: Examined trading patterns and price movements
- **Reporting**: Generated comprehensive reports and exported results

All analysis functions are modularized in the `helpers` folder for reusability and maintainability.